In [1]:
from vcm_model import  *

In [2]:
##评估指标说明：
##每个指标对应的list按照如下顺序输入：
##第一个元素为指标类型，type1:原子指标；type2:策略不影响分母；type3:策略影响分母
##第二个元素为指标分子（原子指标即为指标本身）
##第三个元素为分子的协变量，list形式，无协变量则置空
##第四个元素为指标分母
##第五个元素为分母的协变量，list形式，无协变量则置空
eval_dict={
    '冒泡数':['type1','bubble_cnt',[]],
    '发单数':['type1','compete_call_cnt_hxz',[]],
    '完单数':['type1','hxz_finish_cnt',['compete_call_cnt_hxz','valid_tsh']],
    '应答率':['type3','hxz_answer_cnt',['compete_call_cnt_hxz','valid_tsh'],'compete_call_cnt_hxz',['compete_call_cnt_hxz','valid_tsh']],
    '成交率':['type3','hxz_finish_cnt',['compete_call_cnt_hxz','valid_tsh'],'compete_call_cnt_hxz',['compete_call_cnt_hxz','valid_tsh']],
    '应答后司机取消率(含改派)':['type3','hxz_grab_after_dri_cancel_cnt',['compete_call_cnt_hxz','valid_tsh'],'hxz_answer_cnt',['compete_call_cnt_hxz','valid_tsh']],
    '应答后司机取消率(不含改派)':['type3','hxz_grab_after_dri_cancel_cnt_without_reassign',['compete_call_cnt_hxz','valid_tsh'],'hxz_answer_cnt',['compete_call_cnt_hxz','valid_tsh']],
    '应答后乘客取消率':['type3','hxz_grab_after_pas_cancel_cnt',['compete_call_cnt_hxz','valid_tsh'],'hxz_answer_cnt',['compete_call_cnt_hxz','valid_tsh']],
    '自营完单数':['type1','hxz_zy_finish_cnt',['compete_call_cnt_hxz','valid_tsh']],
    '宏鹄完单数':['type1','hxz_hh_finish_cnt',['compete_call_cnt_hxz','valid_tsh']],
    'API完单数':['type1','hxz_api_finish_cnt',['compete_call_cnt_hxz','valid_tsh']]
}

In [3]:
##评估指标说明：
##每个指标对应的list按照如下顺序输入：
##第一个元素为指标类型，type1:原子指标；type2:策略不影响分母；type3:策略影响分母
##第二个元素为指标分子（原子指标即为指标本身）
##第三个元素为分子的协变量，list形式，无协变量则置空
##第四个元素为指标分母
##第五个元素为分母的协变量，list形式，无协变量则置空
# eval_dict={
#     'GMV':['type1','gmv',['发单量','TSH'],],
#     '单均GMV':['type2','gmv',['TSH'],'发单量'],
#     '应答后取消率':['type3','应答后取消量',[],'应答量',['发单量','TSH']],
# }

In [4]:
##数据格式说明：
##city_id：城市id，必须项
##tdate：实验日期，必须项
##exp_group：实验分组情况，枚举值treatment_group、control_group，必须项
##minf：时间片标识，必须项
##以上必须项的字段名称和实验分组名称，需与demo数据保持一致
df = pd.read_csv('data3.csv')
def chenge_group_name(x):
    return x + '_group'
df['exp_group'] = df['ab_group'].apply(chenge_group_name)
df.head()

,city_id,tdate,minf,ab_group,hxz_finish_cnt,hxz_answer_cnt,hxz_answer_cnt_without_reassign,hxz_grab_after_dri_cancel_cnt,hxz_grab_after_dri_cancel_cnt_without_reassign,hxz_grab_after_pas_cancel_cnt,hxz_grab_after_pas_cancel_cnt_without_reassign,hxz_gmv_amt,hxz_zy_answer_cnt,hxz_zy_finish_cnt,hxz_hh_answer_cnt,hxz_hh_finish_cnt,hxz_api_finish_cnt,pbd_answer_cnt,pbd_finish_cnt,pbd_zy_answer_cnt,pbd_zy_finish_cnt,pbd_hh_answer_cnt,pbd_hh_finish_cnt,pbd_api_finish_cnt,thlm_answer_cnt,thlm_finish_cnt,thlm_zy_answer_cnt,thlm_zy_finish_cnt,thlm_hh_answer_cnt,thlm_hh_finish_cnt,thlm_api_finish_cnt,juhe_zyhh_finish_cnt,juhe_zy_finish_cnt,juhe_zy_gmv_amt,compete_call_cnt_unity,compete_call_cnt_hxz,compete_call_cnt_pbd,compete_call_cnt_zy,compete_call_cnt_hh,compete_call_cnt_zyhh,compete_call_cnt_sf,compete_call_cnt_hxz_zy,compete_call_cnt_hxz_hh,compete_call_cnt_hxz_sf,compete_call_cnt_pbd_zy,compete_call_cnt_pbd_hh,compete_call_cnt_pbd_sf,bubble_cnt,valid_tsh,exp_group
0,4,2025-03-18,5,control,138,178,174,12,8,26,26,4045.71000,0,0,6,4,56,261,203,0,0,7,5,14,637,506,0,0,32,20,486,29,0,NaN,4623,194,1181,0,2367,2367,4203,0,156,186,0,284,1087,338,79.683333,control_group
1,4,2025-03-18,6,treatment,252,319,310,17,6,46,46,8925.61000,0,0,36,18,114,420,316,0,0,78,44,46,2216,1770,0,0,333,197,1573,259,0,NaN,13721,364,2184,0,7315,7315,12529,0,246,333,0,625,2025,648,229.083333,treatment_group
2,4,2025-03-18,7,control,474,604,585,22,4,106,104,14979.44424,0,0,59,33,225,521,400,0,0,43,22,77,3475,2570,0,0,606,253,2317,308,0,NaN,19846,671,2171,0,7643,7643,18679,0,268,640,0,323,2106,1369,506.183333,control_group
3,4,2025-03-18,8,treatment,622,842,814,37,10,177,173,20708.62135,0,0,69,35,289,663,473,0,0,87,42,90,4418,3242,0,0,733,304,2938,381,0,NaN,34899,957,4351,0,15651,15651,33240,0,493,922,0,1035,4200,1802,475.166667,treatment_group
4,4,2025-03-18,11,control,534,638,623,23,5,78,77,16654.47424,0,0,73,40,330,486,379,0,0,61,29,54,3687,2964,0,0,337,175,2789,244,0,NaN,16902,703,2477,0,9303,9303,14130,0,577,617,0,477,2335,1437,339.483333,control_group


In [5]:
##评估结果将自动输出到该目录下的Result文件夹中
import warnings
warnings.filterwarnings("ignore")
vcm = vcm_result(data=df,evaluate_dict=eval_dict)
vcm.run()

  0%|          | 0/13 [00:00<?, ?it/s]

4



  0%|          | 0/11 [00:00<?, ?it/s]

冒泡数



  9%|▉         | 1/11 [00:00<00:06,  1.52it/s]

发单数


... (304 outputs left out)


 91%|█████████ | 10/11 [00:13<00:01,  1.35s/it]

API完单数



100%|██████████| 13/13 [03:25<00:00, 15.78s/it]


,指标,城市,对照组,实验组,P值,绝对提升,相对提升,绝对提升置信区间下界,绝对提升置信区间上界
0,冒泡数,整体,346528.857143,346516.858000,9.886864e-01,-11.999143,-0.000035,-1.670551e+03,1.646553e+03
1,发单数,整体,240613.857143,240227.313325,5.588909e-01,-386.543818,-0.001606,-1.682762e+03,9.096740e+02
2,完单数,整体,173347.714286,173501.951939,4.635635e-01,154.237654,0.000890,-2.581897e+02,5.666650e+02
3,应答率,整体,0.930643,0.927894,4.501534e-02,-0.002749,-0.002954,NaN,NaN
4,成交率,整体,0.720439,0.721080,2.632215e-01,0.000641,0.000890,5.398367e-04,7.421969e-04
5,应答后司机取消率(含改派),整体,0.077333,0.076552,3.423414e-02,-0.000782,-0.010107,-5.424468e-02,5.268151e-02
6,应答后司机取消率(不含改派),整体,0.009127,0.009372,5.061841e-02,0.000245,0.026817,-6.301099e-03,6.790643e-03
7,应答后乘客取消率,整体,0.144657,0.143003,2.002702e-03,-0.001654,-0.011434,-1.015277e-01,9.821977e-02
8,自营完单数,整体,47554.571429,47710.185065,2.308225e-01,155.613637,0.003272,-9.892801e+01,4.101553e+02
9,宏鹄完单数,整体,70793.285714,68887.449900,0.000000e+00,-1905.835814,-0.026921,-2.235346e+03,-1.576325e+03
